In [0]:
MY_ID = "nisha"

VOL = f"/Volumes/workspace/capstone_{MY_ID}/raw"

print(MY_ID)
print(VOL)

nisha
/Volumes/workspace/capstone_nisha/raw


In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.capstone_nisha;

CREATE VOLUME IF NOT EXISTS workspace.capstone_nisha.raw;

In [0]:
%sql
DESCRIBE VOLUME workspace.capstone_nisha.raw;

name,catalog,database,owner,storage_location,volume_type,comment,securable_type,securable_kind
raw,workspace,capstone_nisha,23053630@kiit.ac.in,,MANAGED,null,VOLUME,VOLUME_DB_STORAGE


In [0]:
import pyspark.sql.functions as F
from pyspark.sql import Window

CI = "element_at(array('Pune','Kochi','Indore','Jaipur','Nagpur','Surat'),cast(id%6 as int)+1)"
PK, MD = '(9,10,11,12,17,18,19,20)', '(6,7,8,13,14,15,16,21)'

t = spark.range(120).selectExpr(
    'id n',
    "format_string('T%03d',id+1) tower_id",
    f"concat({CI},' Site ',format_string('%03d',id+1)) site_name",
    f'{CI} city',
    "format_string('D%d',cast(id%4 as int)+1) district",
    'if(id<80,10,2) capacity_channels',
    "date_add(date'2019-01-01',cast(id*11 as int)) commissioned_on"
)

t.drop('n').write.mode('overwrite').option('header',True).csv(
    f'{VOL}/raw/towers'
)

c = (
    t.withColumn(
        'call_date',
        F.explode(
            F.expr("sequence(date'2025-06-01',date'2025-06-15')")
        )
    )
    .withColumn(
        'hour_of_day',
        F.explode(F.expr('sequence(0,23)'))
    )
    .selectExpr(
        'n',
        'tower_id',
        'call_date',
        'hour_of_day',
        'if(n<80,6,1) reps',
        f'case when hour_of_day in {PK} then 3 when hour_of_day in {MD} then 2 else 1 end w'
    )
    .selectExpr(
        '*',
        'w*reps cnt',
        'if(n<80,element_at(array(360,180,72),w),element_at(array(20,10,6),w)) md'
    )
)

c = (
    c.withColumn(
        'k',
        F.explode(F.expr('sequence(1,cnt)'))
    )
    .selectExpr(
        '*',
        'hour_of_day*3600+(k-1)*cast(3600/cnt as int) ss',
        'element_at(array(45,240,420),w)+(k%4)*element_at(array(15,120,120),w) dur',
        "concat_ws('-',tower_id,string(call_date),string(hour_of_day),string(k)) call_id"
    )
    .withColumn(
        'rb',
        F.row_number().over(
            Window.partitionBy('tower_id','w')
            .orderBy('call_date','hour_of_day','k')
        )
    )
)

c = (
    c.selectExpr(
        '*',
        "case when rb%md=0 then 'DROPPED' "
        "when rb%17=1 then 'BUSY' "
        "when rb%23=2 then 'NO_ANSWER' "
        "else 'NORMAL' end cause"
    )
    .withColumn(
        'rn',
        F.row_number().over(Window.orderBy('call_id'))
    )
)

raw = c.selectExpr(
    'rn',
    'call_id',
    "format_string('SUB%05d',cast(pmod(k*37+hour_of_day*11+n*7,60000) as int)) subscriber_id",
    "if(rn%624=2,'T999',tower_id) tower_id",
    "cast(timestampadd(second,ss,cast(call_date as timestamp)) as string) start_ts",
    "case when rn%312=0 then 'NA' when rn%832=3 then '-1' else string(dur) end duration_seconds",
    "if(rn%416=1,concat(' ',lower(cause),' '),cause) end_cause",
    "if(k%2=0,'MT','MO') direction",
    "if(n%3=0,'5G','4G') technology"
)

(
    raw.unionByName(raw.filter('rn%208=5'))
    .drop('rn')
    .write.mode('overwrite')
    .option('header',True)
    .option('ignoreLeadingWhiteSpace',False)
    .option('ignoreTrailingWhiteSpace',False)
    .csv(f'{VOL}/raw/calls')
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
towers = spark.read.csv(
    f"{VOL}/raw/towers",
    header=True,
    inferSchema=True
)

calls = spark.read.csv(
    f"{VOL}/raw/calls",
    header=True,
    inferSchema=True
)

print("Towers:", towers.count())
print("Calls:", calls.count())

Towers: 120
Calls: 376200


In [0]:
from pyspark.sql import functions as F

# ---------------------------------------------------------
# BRONZE — LAND RAW DATA WITHOUT CLEANING
# ---------------------------------------------------------

# Read every raw column as STRING
towers_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .csv(f"{VOL}/raw/towers")
)

calls_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .csv(f"{VOL}/raw/calls")
)

# ---------------------------------------------------------
# Add Bronze provenance columns
# Unity Catalog: use _metadata.file_path
# ---------------------------------------------------------

bronze_towers = (
    towers_raw
    .withColumn("_source_file", F.col("_metadata.file_path"))
    .withColumn("_ingested_at", F.current_timestamp())
)

bronze_calls = (
    calls_raw
    .withColumn("_source_file", F.col("_metadata.file_path"))
    .withColumn("_ingested_at", F.current_timestamp())
)

# ---------------------------------------------------------
# Row hashes over raw columns
# ---------------------------------------------------------

tower_cols = towers_raw.columns
call_cols = calls_raw.columns

bronze_towers = bronze_towers.withColumn(
    "_row_hash",
    F.sha2(
        F.concat_ws("||", *[F.col(c) for c in tower_cols]),
        256
    )
)

bronze_calls = bronze_calls.withColumn(
    "_row_hash",
    F.sha2(
        F.concat_ws("||", *[F.col(c) for c in call_cols]),
        256
    )
)

# ---------------------------------------------------------
# Bronze validation
# ---------------------------------------------------------

print("Bronze towers:", bronze_towers.count())
print("Bronze calls:", bronze_calls.count())

print("Tower schema:")
bronze_towers.printSchema()

print("Call schema:")
bronze_calls.printSchema()

Bronze towers: 120
Bronze calls: 376200
Tower schema:
root
 |-- tower_id: string (nullable = true)
 |-- site_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- district: string (nullable = true)
 |-- capacity_channels: string (nullable = true)
 |-- commissioned_on: string (nullable = true)
 |-- _source_file: string (nullable = false)
 |-- _ingested_at: timestamp (nullable = false)
 |-- _row_hash: string (nullable = true)

Call schema:
root
 |-- call_id: string (nullable = true)
 |-- subscriber_id: string (nullable = true)
 |-- tower_id: string (nullable = true)
 |-- start_ts: string (nullable = true)
 |-- duration_seconds: string (nullable = true)
 |-- end_cause: string (nullable = true)
 |-- direction: string (nullable = true)
 |-- technology: string (nullable = true)
 |-- _source_file: string (nullable = false)
 |-- _ingested_at: timestamp (nullable = false)
 |-- _row_hash: string (nullable = true)



In [0]:
bronze_towers.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.capstone_nisha.bronze_towers")

bronze_calls.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.capstone_nisha.bronze_calls")

print("Bronze towers:",
      spark.table("workspace.capstone_nisha.bronze_towers").count())

print("Bronze calls:",
      spark.table("workspace.capstone_nisha.bronze_calls").count())

Bronze towers: 120
Bronze calls: 376200


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ---------------------------------------------------------
# SILVER — CLEAN, VALIDATE, DEDUPLICATE, AND REJECT
# ---------------------------------------------------------

# Read persisted Bronze tables
bronze_calls_df = spark.table(
    "workspace.capstone_nisha.bronze_calls"
)

bronze_towers_df = spark.table(
    "workspace.capstone_nisha.bronze_towers"
)

# ---------------------------------------------------------
# 1. Safe type conversions
# ---------------------------------------------------------

s = (
    bronze_calls_df
    .withColumn(
        "duration_seconds_int",
        F.expr("try_cast(duration_seconds AS INT)")
    )
    .withColumn(
        "start_ts_clean",
        F.expr("try_cast(start_ts AS TIMESTAMP)")
    )
)

# ---------------------------------------------------------
# 2. Deduplicate by call_id
# ---------------------------------------------------------

w_dedupe = (
    Window
    .partitionBy("call_id")
    .orderBy(F.col("_ingested_at"))
)

s = (
    s
    .withColumn("_rn", F.row_number().over(w_dedupe))
    .filter(F.col("_rn") == 1)
    .drop("_rn")
)

# ---------------------------------------------------------
# 3. Clean end_cause
# ---------------------------------------------------------

s = s.withColumn(
    "end_cause_clean",
    F.upper(F.trim(F.col("end_cause")))
)

# ---------------------------------------------------------
# 4. Derive is_dropped
# ---------------------------------------------------------

s = s.withColumn(
    "is_dropped",
    F.when(
        F.col("end_cause_clean") == "DROPPED",
        1
    ).otherwise(0)
)

# ---------------------------------------------------------
# 5. Join tower reference data
# ---------------------------------------------------------

tower_lookup = bronze_towers_df.select(
    "tower_id",
    "site_name",
    "city",
    "district",
    "capacity_channels"
)

s = s.join(
    tower_lookup,
    on="tower_id",
    how="left"
)

# ---------------------------------------------------------
# 6. Assign rejection reason
# ---------------------------------------------------------

s = s.withColumn(
    "reject_reason",
    F.when(
        F.col("site_name").isNull(),
        F.lit("unknown_tower")
    )
    .when(
        F.col("duration_seconds_int").isNull(),
        F.lit("duration_not_numeric")
    )
    .when(
        (F.col("duration_seconds_int") < 0) |
        (F.col("duration_seconds_int") > 7200),
        F.lit("duration_out_of_range")
    )
    .when(
        F.col("start_ts_clean").isNull(),
        F.lit("start_ts_not_timestamp")
    )
)

# ---------------------------------------------------------
# 7. Create Silver rejects
# ---------------------------------------------------------

silver_rejects = (
    s
    .filter(F.col("reject_reason").isNotNull())
)

# ---------------------------------------------------------
# 8. Create valid Silver calls
# ---------------------------------------------------------

silver_calls = (
    s
    .filter(F.col("reject_reason").isNull())
    .drop(
        "reject_reason",
        "duration_seconds",
        "start_ts"
    )
    .withColumnRenamed(
        "duration_seconds_int",
        "duration_seconds"
    )
    .withColumnRenamed(
        "start_ts_clean",
        "start_ts"
    )
)

# ---------------------------------------------------------
# 9. Derive end_ts in Silver
# ---------------------------------------------------------

silver_calls = silver_calls.withColumn(
    "end_ts",
    F.expr(
        "start_ts + INTERVAL 1 SECOND * duration_seconds"
    )
)

# ---------------------------------------------------------
# 10. Silver validation
# ---------------------------------------------------------

print("Silver rows:", silver_calls.count())
print("Rejected rows:", silver_rejects.count())

print("Silver schema:")
silver_calls.printSchema()

print("Reject reasons:")
silver_rejects.groupBy(
    "reject_reason"
).count().orderBy(
    "reject_reason"
).show(truncate=False)

Silver rows: 372150
Rejected rows: 2250
Silver schema:
root
 |-- tower_id: string (nullable = true)
 |-- call_id: string (nullable = true)
 |-- subscriber_id: string (nullable = true)
 |-- end_cause: string (nullable = true)
 |-- direction: string (nullable = true)
 |-- technology: string (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _row_hash: string (nullable = true)
 |-- duration_seconds: integer (nullable = true)
 |-- start_ts: timestamp (nullable = true)
 |-- end_cause_clean: string (nullable = true)
 |-- is_dropped: integer (nullable = false)
 |-- site_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- district: string (nullable = true)
 |-- capacity_channels: string (nullable = true)
 |-- end_ts: timestamp (nullable = true)

Reject reasons:
+---------------------+-----+
|reject_reason        |count|
+---------------------+-----+
|duration_not_numeric |1200 |
|duration_out_of_range|450  |
|un

In [0]:
# ---------------------------------------------------------
# SILVER — PERSIST TABLES
# ---------------------------------------------------------

silver_calls.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "workspace.capstone_nisha.silver_calls"
    )

silver_rejects.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "workspace.capstone_nisha.silver_rejects"
    )

print(
    "Silver calls:",
    spark.table(
        "workspace.capstone_nisha.silver_calls"
    ).count()
)

print(
    "Silver rejects:",
    spark.table(
        "workspace.capstone_nisha.silver_rejects"
    ).count()
)

Silver calls: 372150
Silver rejects: 2250


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ---------------------------------------------------------
# GOLD — BUILD TOWER / DATE / HOUR AGGREGATION
# ---------------------------------------------------------

# 1. Read the finalized persisted Silver table
silver_for_gold = spark.table(
    "workspace.capstone_nisha.silver_calls"
)

# ---------------------------------------------------------
# 2. Prepare date and hour fields
#    start_ts and end_ts were already cleaned in Silver
# ---------------------------------------------------------

gold_base = (
    silver_for_gold
    .withColumn("call_date", F.to_date("start_ts"))
    .withColumn("hour_of_day", F.hour("start_ts"))
)

# ---------------------------------------------------------
# 3. Aggregate calls by tower / date / hour
# ---------------------------------------------------------

gold_counts = (
    gold_base
    .groupBy(
        "tower_id",
        "site_name",
        "city",
        "district",
        "capacity_channels",
        "call_date",
        "hour_of_day"
    )
    .agg(
        F.count("*").alias("calls"),
        F.sum("is_dropped").alias("dropped_calls"),
        F.avg("duration_seconds").alias("avg_duration_seconds")
    )
)

# ---------------------------------------------------------
# 4. Calculate drop rate
#    Ratio of sums: dropped_calls / calls
# ---------------------------------------------------------

gold_counts = gold_counts.withColumn(
    "drop_rate",
    F.when(
        F.col("calls") > 0,
        F.col("dropped_calls") / F.col("calls")
    ).otherwise(F.lit(0.0))
)

# ---------------------------------------------------------
# 5. Create concurrency start/end events
# ---------------------------------------------------------

starts = gold_base.select(
    "tower_id",
    "call_date",
    "hour_of_day",
    F.col("start_ts").alias("event_ts"),
    F.lit(1).alias("delta")
)

ends = gold_base.select(
    "tower_id",
    "call_date",
    "hour_of_day",
    F.col("end_ts").alias("event_ts"),
    F.lit(-1).alias("delta")
)

events = starts.unionByName(ends)

# ---------------------------------------------------------
# 6. Calculate running concurrent calls
# ---------------------------------------------------------

w_concurrent = (
    Window
    .partitionBy(
        "tower_id",
        "call_date",
        "hour_of_day"
    )
    .orderBy(
        "event_ts",
        "delta"
    )
    .rowsBetween(
        Window.unboundedPreceding,
        Window.currentRow
    )
)

events = events.withColumn(
    "concurrent",
    F.sum("delta").over(w_concurrent)
)

# ---------------------------------------------------------
# 7. Peak concurrency per tower / date / hour
# ---------------------------------------------------------

peak = (
    events
    .groupBy(
        "tower_id",
        "call_date",
        "hour_of_day"
    )
    .agg(
        F.max("concurrent").alias("peak_concurrent")
    )
)

# ---------------------------------------------------------
# 8. Build final Gold dataframe
# ---------------------------------------------------------

gold = (
    gold_counts
    .join(
        peak,
        on=[
            "tower_id",
            "call_date",
            "hour_of_day"
        ],
        how="left"
    )
    .withColumn(
        "utilisation",
        F.col("peak_concurrent") /
        F.col("capacity_channels").cast("double")
    )
)

# ---------------------------------------------------------
# 9. Gold validation
# ---------------------------------------------------------

print("Gold rows:", gold.count())

print(
    "Gold calls:",
    gold.agg(
        F.sum("calls")
    ).first()[0]
)

print(
    "Gold dropped:",
    gold.agg(
        F.sum("dropped_calls")
    ).first()[0]
)

print("Gold schema:")
gold.printSchema()

Gold rows: 43142
Gold calls: 372150
Gold dropped: 6792
Gold schema:
root
 |-- tower_id: string (nullable = true)
 |-- call_date: date (nullable = true)
 |-- hour_of_day: integer (nullable = true)
 |-- site_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- district: string (nullable = true)
 |-- capacity_channels: string (nullable = true)
 |-- calls: long (nullable = false)
 |-- dropped_calls: long (nullable = true)
 |-- avg_duration_seconds: double (nullable = true)
 |-- drop_rate: double (nullable = true)
 |-- peak_concurrent: long (nullable = true)
 |-- utilisation: double (nullable = true)



In [0]:
# ---------------------------------------------------------
# FINAL CONCURRENCY VALIDATION
# ---------------------------------------------------------

print("HEALTHY TOWER T001 - expected pattern")

(
    gold
    .filter(F.col("tower_id") == "T001")
    .groupBy("hour_of_day")
    .agg(
        F.max("peak_concurrent").alias("peak_concurrent")
    )
    .orderBy("hour_of_day")
    .show(24, truncate=False)
)

print("WEAK TOWER T081 - expected 1 everywhere")

(
    gold
    .filter(F.col("tower_id") == "T081")
    .groupBy("hour_of_day")
    .agg(
        F.max("peak_concurrent").alias("peak_concurrent")
    )
    .orderBy("hour_of_day")
    .show(24, truncate=False)
)

HEALTHY TOWER T001 - expected pattern
+-----------+---------------+
|hour_of_day|peak_concurrent|
+-----------+---------------+
|0          |1              |
|1          |1              |
|2          |1              |
|3          |1              |
|4          |1              |
|5          |1              |
|6          |2              |
|7          |2              |
|8          |2              |
|9          |4              |
|10         |4              |
|11         |4              |
|12         |4              |
|13         |2              |
|14         |2              |
|15         |2              |
|16         |2              |
|17         |4              |
|18         |4              |
|19         |4              |
|20         |4              |
|21         |2              |
|22         |1              |
|23         |1              |
+-----------+---------------+

WEAK TOWER T081 - expected 1 everywhere
+-----------+---------------+
|hour_of_day|peak_concurrent|
+-----------+--------

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ---------------------------------------------------------
# GOLD — BUILD TOWER / DATE / HOUR AGGREGATION
# ---------------------------------------------------------

# 1. Read the finalized persisted Silver table
silver_for_gold = spark.table(
    "workspace.capstone_nisha.silver_calls"
)

# ---------------------------------------------------------
# 2. Prepare date and hour fields
#    start_ts and end_ts were already cleaned in Silver
# ---------------------------------------------------------

gold_base = (
    silver_for_gold
    .withColumn("call_date", F.to_date("start_ts"))
    .withColumn("hour_of_day", F.hour("start_ts"))
)

# ---------------------------------------------------------
# 3. Aggregate calls by tower / date / hour
# ---------------------------------------------------------

gold_counts = (
    gold_base
    .groupBy(
        "tower_id",
        "site_name",
        "city",
        "district",
        "capacity_channels",
        "call_date",
        "hour_of_day"
    )
    .agg(
        F.count("*").alias("calls"),
        F.sum("is_dropped").alias("dropped_calls"),
        F.avg("duration_seconds").alias("avg_duration_seconds")
    )
)

# ---------------------------------------------------------
# 4. Calculate drop rate
#    Ratio of sums: dropped_calls / calls
# ---------------------------------------------------------

gold_counts = gold_counts.withColumn(
    "drop_rate",
    F.when(
        F.col("calls") > 0,
        F.col("dropped_calls") / F.col("calls")
    ).otherwise(F.lit(0.0))
)

# ---------------------------------------------------------
# 5. Create concurrency start/end events
# ---------------------------------------------------------

starts = gold_base.select(
    "tower_id",
    "call_date",
    "hour_of_day",
    F.col("start_ts").alias("event_ts"),
    F.lit(1).alias("delta")
)

ends = gold_base.select(
    "tower_id",
    "call_date",
    "hour_of_day",
    F.col("end_ts").alias("event_ts"),
    F.lit(-1).alias("delta")
)

events = starts.unionByName(ends)

# ---------------------------------------------------------
# 6. Calculate running concurrent calls
# ---------------------------------------------------------

w_concurrent = (
    Window
    .partitionBy(
        "tower_id",
        "call_date",
        "hour_of_day"
    )
    .orderBy(
        "event_ts",
        "delta"
    )
    .rowsBetween(
        Window.unboundedPreceding,
        Window.currentRow
    )
)

events = events.withColumn(
    "concurrent",
    F.sum("delta").over(w_concurrent)
)

# ---------------------------------------------------------
# 7. Peak concurrency per tower / date / hour
# ---------------------------------------------------------

peak = (
    events
    .groupBy(
        "tower_id",
        "call_date",
        "hour_of_day"
    )
    .agg(
        F.max("concurrent").alias("peak_concurrent")
    )
)

# ---------------------------------------------------------
# 8. Build final Gold dataframe
# ---------------------------------------------------------

gold = (
    gold_counts
    .join(
        peak,
        on=[
            "tower_id",
            "call_date",
            "hour_of_day"
        ],
        how="left"
    )
    .withColumn(
        "utilisation",
        F.col("peak_concurrent") /
        F.col("capacity_channels").cast("double")
    )
)

# ---------------------------------------------------------
# 9. Gold validation
# ---------------------------------------------------------

print("Gold rows:", gold.count())

print(
    "Gold calls:",
    gold.agg(
        F.sum("calls")
    ).first()[0]
)

print(
    "Gold dropped:",
    gold.agg(
        F.sum("dropped_calls")
    ).first()[0]
)

print("Gold schema:")
gold.printSchema()

Gold rows: 43142
Gold calls: 372150
Gold dropped: 6792
Gold schema:
root
 |-- tower_id: string (nullable = true)
 |-- call_date: date (nullable = true)
 |-- hour_of_day: integer (nullable = true)
 |-- site_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- district: string (nullable = true)
 |-- capacity_channels: string (nullable = true)
 |-- calls: long (nullable = false)
 |-- dropped_calls: long (nullable = true)
 |-- avg_duration_seconds: double (nullable = true)
 |-- drop_rate: double (nullable = true)
 |-- peak_concurrent: long (nullable = true)
 |-- utilisation: double (nullable = true)



In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ---------------------------------------------------------
# GOLD — BUILD TOWER / DATE / HOUR AGGREGATION
# ---------------------------------------------------------

# 1. Read the finalized persisted Silver table
silver_for_gold = spark.table(
    "workspace.capstone_nisha.silver_calls"
)

# ---------------------------------------------------------
# 2. Prepare date and hour fields
#    start_ts and end_ts were already cleaned in Silver
# ---------------------------------------------------------

gold_base = (
    silver_for_gold
    .withColumn("call_date", F.to_date("start_ts"))
    .withColumn("hour_of_day", F.hour("start_ts"))
)

# ---------------------------------------------------------
# 3. Aggregate calls by tower / date / hour
# ---------------------------------------------------------

gold_counts = (
    gold_base
    .groupBy(
        "tower_id",
        "site_name",
        "city",
        "district",
        "capacity_channels",
        "call_date",
        "hour_of_day"
    )
    .agg(
        F.count("*").alias("calls"),
        F.sum("is_dropped").alias("dropped_calls"),
        F.avg("duration_seconds").alias("avg_duration_seconds")
    )
)

# ---------------------------------------------------------
# 4. Calculate drop rate
#    Ratio of sums: dropped_calls / calls
# ---------------------------------------------------------

gold_counts = gold_counts.withColumn(
    "drop_rate",
    F.when(
        F.col("calls") > 0,
        F.col("dropped_calls") / F.col("calls")
    ).otherwise(F.lit(0.0))
)

# ---------------------------------------------------------
# 5. Create concurrency start/end events
# ---------------------------------------------------------

starts = gold_base.select(
    "tower_id",
    "call_date",
    "hour_of_day",
    F.col("start_ts").alias("event_ts"),
    F.lit(1).alias("delta")
)

ends = gold_base.select(
    "tower_id",
    "call_date",
    "hour_of_day",
    F.col("end_ts").alias("event_ts"),
    F.lit(-1).alias("delta")
)

events = starts.unionByName(ends)

# ---------------------------------------------------------
# 6. Calculate running concurrent calls
# ---------------------------------------------------------

w_concurrent = (
    Window
    .partitionBy(
        "tower_id",
        "call_date",
        "hour_of_day"
    )
    .orderBy(
        "event_ts",
        "delta"
    )
    .rowsBetween(
        Window.unboundedPreceding,
        Window.currentRow
    )
)

events = events.withColumn(
    "concurrent",
    F.sum("delta").over(w_concurrent)
)

# ---------------------------------------------------------
# 7. Peak concurrency per tower / date / hour
# ---------------------------------------------------------

peak = (
    events
    .groupBy(
        "tower_id",
        "call_date",
        "hour_of_day"
    )
    .agg(
        F.max("concurrent").alias("peak_concurrent")
    )
)

# ---------------------------------------------------------
# 8. Build final Gold dataframe
# ---------------------------------------------------------

gold = (
    gold_counts
    .join(
        peak,
        on=[
            "tower_id",
            "call_date",
            "hour_of_day"
        ],
        how="left"
    )
    .withColumn(
        "utilisation",
        F.col("peak_concurrent") /
        F.col("capacity_channels").cast("double")
    )
)

# ---------------------------------------------------------
# 9. Gold validation
# ---------------------------------------------------------

print("Gold rows:", gold.count())

print(
    "Gold calls:",
    gold.agg(
        F.sum("calls")
    ).first()[0]
)

print(
    "Gold dropped:",
    gold.agg(
        F.sum("dropped_calls")
    ).first()[0]
)

print("Gold schema:")
gold.printSchema()

Gold rows: 43142
Gold calls: 372150
Gold dropped: 6792
Gold schema:
root
 |-- tower_id: string (nullable = true)
 |-- call_date: date (nullable = true)
 |-- hour_of_day: integer (nullable = true)
 |-- site_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- district: string (nullable = true)
 |-- capacity_channels: string (nullable = true)
 |-- calls: long (nullable = false)
 |-- dropped_calls: long (nullable = true)
 |-- avg_duration_seconds: double (nullable = true)
 |-- drop_rate: double (nullable = true)
 |-- peak_concurrent: long (nullable = true)
 |-- utilisation: double (nullable = true)



In [0]:
# ---------------------------------------------------------
# GOLD QUESTION 1
# Which hour is worst for each tower?
# ---------------------------------------------------------

from pyspark.sql import functions as F
from pyspark.sql.window import Window

tower_hour = (
    gold
    .groupBy(
        "tower_id",
        "hour_of_day"
    )
    .agg(
        F.sum("calls").alias("calls"),
        F.sum("dropped_calls").alias("drops")
    )
    .withColumn(
        "drop_rate",
        F.when(
            F.col("calls") > 0,
            F.col("drops") / F.col("calls")
        ).otherwise(F.lit(0.0))
    )
)

worst_hour = (
    tower_hour
    .withColumn(
        "tower_drop_rate",
        F.sum("drops").over(
            Window.partitionBy("tower_id")
        ) /
        F.sum("calls").over(
            Window.partitionBy("tower_id")
        )
    )
    .withColumn(
        "rn",
        F.row_number().over(
            Window
            .partitionBy("tower_id")
            .orderBy(
                F.col("drop_rate").desc(),
                F.col("hour_of_day")
            )
        )
    )
    .filter(
        F.col("rn") == 1
    )
    .select(
        "tower_id",
        "hour_of_day",
        "drop_rate",
        "tower_drop_rate"
    )
)

print("Q1 — Worst hour for each tower")

worst_hour \
    .orderBy("tower_id") \
    .show(120, truncate=False)

Q1 — Worst hour for each tower
+--------+-----------+--------------------+--------------------+
|tower_id|hour_of_day|drop_rate           |tower_drop_rate     |
+--------+-----------+--------------------+--------------------+
|T001    |12         |0.055762081784386616|0.009315323707498836|
|T002    |12         |0.055970149253731345|0.009315323707498836|
|T003    |12         |0.055762081784386616|0.009315323707498836|
|T004    |12         |0.055762081784386616|0.009315323707498836|
|T005    |20         |0.055970149253731345|0.009082440614811364|
|T006    |12         |0.055762081784386616|0.009317493594223155|
|T007    |12         |0.055762081784386616|0.009313154831199068|
|T008    |12         |0.055970149253731345|0.009315323707498836|
|T009    |12         |0.055762081784386616|0.009315323707498836|
|T010    |12         |0.055762081784386616|0.009315323707498836|
|T011    |20         |0.055970149253731345|0.009317493594223155|
|T012    |12         |0.055762081784386616|0.00931532370749

In [0]:
# ---------------------------------------------------------
# FINAL CONCURRENCY CHECK
# One result per hour across all 15 days
# ---------------------------------------------------------

print("HEALTHY TOWER T001 - expected: busy=4, medium=2, quiet=1")

(
    gold
    .filter(F.col("tower_id") == "T001")
    .groupBy("hour_of_day")
    .agg(
        F.max("peak_concurrent").alias("peak_concurrent")
    )
    .orderBy("hour_of_day")
    .show(24, False)
)

print("WEAK TOWER T081 - expected: 1 everywhere")

(
    gold
    .filter(F.col("tower_id") == "T081")
    .groupBy("hour_of_day")
    .agg(
        F.max("peak_concurrent").alias("peak_concurrent")
    )
    .orderBy("hour_of_day")
    .show(24, False)
)

HEALTHY TOWER T001 - expected: busy=4, medium=2, quiet=1
+-----------+---------------+
|hour_of_day|peak_concurrent|
+-----------+---------------+
|0          |1              |
|1          |1              |
|2          |1              |
|3          |1              |
|4          |1              |
|5          |1              |
|6          |2              |
|7          |2              |
|8          |2              |
|9          |4              |
|10         |4              |
|11         |4              |
|12         |4              |
|13         |2              |
|14         |2              |
|15         |2              |
|16         |2              |
|17         |4              |
|18         |4              |
|19         |4              |
|20         |4              |
|21         |2              |
|22         |1              |
|23         |1              |
+-----------+---------------+

WEAK TOWER T081 - expected: 1 everywhere
+-----------+---------------+
|hour_of_day|peak_concurrent|
+

In [0]:
# ---------------------------------------------------------
# GOLD QUESTION 2
# Towers exceeding 5% drop rate on more than 10 days
# ---------------------------------------------------------

from pyspark.sql import functions as F

# Calculate daily drop rate for each tower
daily_rates = (
    gold
    .groupBy(
        "tower_id",
        "call_date"
    )
    .agg(
        F.sum("calls").alias("calls"),
        F.sum("dropped_calls").alias("drops")
    )
    .withColumn(
        "drop_rate",
        F.when(
            F.col("calls") > 0,
            F.col("drops") / F.col("calls")
        ).otherwise(F.lit(0.0))
    )
)

# Count days where drop rate exceeded 5%
q2 = (
    daily_rates
    .groupBy("tower_id")
    .agg(
        F.sum(
            F.when(
                F.col("drop_rate") > 0.05,
                1
            ).otherwise(0)
        ).alias("days_over_5_pct")
    )
    .filter(
        F.col("days_over_5_pct") > 10
    )
    .orderBy("tower_id")
)

print("Q2 — Towers over 5% drop rate on more than 10 days")

q2.show(120, truncate=False)

Q2 — Towers over 5% drop rate on more than 10 days
+--------+---------------+
|tower_id|days_over_5_pct|
+--------+---------------+
|T081    |15             |
|T082    |15             |
|T083    |15             |
|T084    |15             |
|T085    |15             |
|T086    |15             |
|T087    |15             |
|T088    |15             |
|T089    |15             |
|T090    |15             |
|T091    |15             |
|T092    |15             |
|T093    |15             |
|T094    |15             |
|T095    |15             |
|T096    |15             |
|T097    |15             |
|T098    |15             |
|T099    |15             |
|T100    |15             |
|T101    |15             |
|T102    |15             |
|T103    |15             |
|T104    |15             |
|T105    |15             |
|T106    |15             |
|T107    |15             |
|T108    |15             |
|T109    |15             |
|T110    |15             |
|T111    |15             |
|T112    |15             |
|T11

In [0]:
# ---------------------------------------------------------
# GOLD QUESTION 3
# Does drop rate rise with concurrent calls?
# ---------------------------------------------------------

from pyspark.sql import functions as F

q3 = (
    gold
    .groupBy("peak_concurrent")
    .agg(
        F.sum("calls").alias("calls"),
        F.sum("dropped_calls").alias("drops")
    )
    .withColumn(
        "drop_rate",
        F.when(
            F.col("calls") > 0,
            F.col("drops") / F.col("calls")
        ).otherwise(F.lit(0.0))
    )
    .orderBy("peak_concurrent")
)

print("Q3 — Drop rate by peak concurrency")

q3.show(truncate=False)

Q3 — Drop rate by peak concurrency
+---------------+------+-----+--------------------+
|peak_concurrent|calls |drops|drop_rate           |
+---------------+------+-----+--------------------+
|1              |85949 |3752 |0.04365379469220119 |
|2              |114370|640  |0.005595873043630323|
|4              |171831|2400 |0.013967211969900658|
+---------------+------+-----+--------------------+



In [0]:
# ---------------------------------------------------------
# SAVE FINAL GOLD TABLE
# ---------------------------------------------------------

gold.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "workspace.capstone_nisha.gold"
    )

print("Gold table saved successfully.")

Gold table saved successfully.


In [0]:
spark.sql(
    "SELECT * FROM workspace.capstone_nisha.gold LIMIT 10"
).show()

+--------+----------+-----------+-------------+----+--------+-----------------+-----+-------------+--------------------+---------+---------------+-----------+
|tower_id| call_date|hour_of_day|    site_name|city|district|capacity_channels|calls|dropped_calls|avg_duration_seconds|drop_rate|peak_concurrent|utilisation|
+--------+----------+-----------+-------------+----+--------+-----------------+-----+-------------+--------------------+---------+---------------+-----------+
|    T001|2025-06-01|          0|Pune Site 001|Pune|      D1|               10|    4|            0|                60.0|      0.0|              1|        0.1|
|    T001|2025-06-01|          1|Pune Site 001|Pune|      D1|               10|    6|            0|                67.5|      0.0|              1|        0.1|
|    T001|2025-06-01|          2|Pune Site 001|Pune|      D1|               10|    6|            0|                67.5|      0.0|              1|        0.1|
|    T001|2025-06-01|          3|Pune Site 001

In [0]:
MY_ID = "nisha"
VOL = f"/Volumes/workspace/capstone_{MY_ID}/raw"

gold.coalesce(1) \
    .write \
    .mode("overwrite") \
    .option("header", True) \
    .csv(f"{VOL}/gold_export")

print("Gold CSV export completed.")
print(f"Location: {VOL}/gold_export")

Gold CSV export completed.
Location: /Volumes/workspace/capstone_nisha/raw/gold_export


In [0]:
gold.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.capstone_nisha.gold")